# Anker-Store-Inspektion

`anchor_store.py` speichert nach *jeder* Pass-1-Analyse automatisch einen neuen Kalibrierungs-Anker (`add_anchor`, ungefiltert, kein Qualitaets-Check). Ab `MIN_ANCHORS=5` werden die `k=3` aehnlichsten Anker dynamisch in den Pass-1-Prompt eingebettet. Das seit [`prompting_patterns.md`](../docs/reference/prompting_patterns.md) dokumentierte, aber **nie empirisch geprüfte** Risiko: ein einzelner Fehlgriff des Modells wird zum Kalibrierungsanker fuer alle aehnlichen kuenftigen Artikel — ein moeglicher Feedback-Loop.

Dieses Notebook macht den Ankerbestand sichtbar, statt das Risiko nur zu vermuten.

**Voraussetzung:** laufende ChromaDB (`chroma run --host localhost --port 8001 --path ../data/chroma_db`).

## 1 — Setup

In [ ]:
import sys
sys.path.insert(0, "../src")

import json
import pandas as pd
from news_analyser.repositories.anchor_store import _get_collection, MIN_ANCHORS, K_RESULTS, get_similar_anchors

col = _get_collection()
n = col.count()
print(f"{n} Anker gespeichert (MIN_ANCHORS={MIN_ANCHORS}, RAG aktiv: {n >= MIN_ANCHORS})")

## 2 — Alle Anker als Tabelle

In [ ]:
def stroemung_labels(raw) -> list[str]:
    # politische_stroemung wurde bei aelteren Ankern als list[str] gespeichert,
    # bei neueren (aktuelles pass2.md-Schema) als JSON-Liste von {label, quote}-Dicts —
    # anchor_store.py dumpt das Feld ungeflattened (Bug, siehe naechste Zelle).
    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except Exception:
            return [raw]
    if not isinstance(raw, list):
        return [str(raw)]
    return [item.get("label", "") if isinstance(item, dict) else str(item) for item in raw]

if n == 0:
    print("Keine Anker vorhanden.")
else:
    result = col.get(include=["metadatas", "documents"])
    rows = []
    for meta, doc in zip(result["metadatas"], result["documents"]):
        rows.append({
            "domain":     meta.get("domain"),
            "orwell":     meta.get("orwell_index"),
            "stroemung":  ", ".join(stroemung_labels(meta.get("politische_stroemung", "[]"))),
            "source_url": meta.get("source_url"),
            "text_len":   len(doc) if doc else 0,
        })
    df_anchors = pd.DataFrame(rows).sort_values("orwell", ascending=False)
    pd.set_option("display.max_colwidth", 60)
    display(df_anchors)

**Bug gefunden beim Bauen dieses Notebooks:** `anchor_store.py:83` speichert
`politische_stroemung` per `json.dumps(politische_stroemung, ...)` ohne vorher auf
Label-Strings zu flatten (der Typehint sagt `list[str]`, tatsaechlich kommt seit dem
aktuellen `pass2.md`-Schema eine `list[dict]` mit `{label, quote}` an —
`analyzer.py` uebergibt `stroemung` ungefiltert an `add_anchor()`). Anders als
`db_storage.py`, das `_extract_stroemung_labels()` vor dem Speichern aufruft, macht
`anchor_store.py` das nicht. Effekt: `format_anchors_for_prompt()` baut die Zeile
`Stroemung: {a['politische_stroemung']}` direkt aus dem rohen Wert — seit dem
dict-basierten Schema landet dort vermutlich ein JSON-Blob
(`[{"label": "neutral", "quote": null}]`) statt eines sauberen Labels im
Kalibrierungstext, der an Pass 1 geht. Noch nicht gefixt — nur hier im Notebook
robust geparst, damit die Tabelle nicht crasht.

## 3 — Verteilung nach Domain

Domains mit vielen Ankern dominieren die RAG-Auswahl fuer thematisch aehnliche kuenftige Artikel derselben Domain (oder aehnlicher Themen) staerker.

In [ ]:
if n > 0:
    display(df_anchors.groupby("domain")["orwell"].agg(["count", "mean", "min", "max"]).sort_values("count", ascending=False))

## 4 — Moegliche Ausreisser

Anker, deren `orwell_index` weit vom Domain-Mittel abweicht — Kandidaten fuer einen Fehlgriff, der sich als Kalibrierungsreferenz weiterverbreiten koennte. `THRESHOLD` ist die Mindestabweichung vom Domain-Mittel.

In [ ]:
THRESHOLD = 0.3

if n > 0:
    domain_mean = df_anchors.groupby("domain")["orwell"].transform("mean")
    df_anchors["deviation"] = (df_anchors["orwell"] - domain_mean).abs()
    outliers = df_anchors[df_anchors["deviation"] > THRESHOLD].sort_values("deviation", ascending=False)
    if outliers.empty:
        print(f"Keine Ausreisser ueber Schwelle {THRESHOLD}.")
    else:
        display(outliers[["domain", "orwell", "deviation", "stroemung", "source_url"]])

## 5 — Qualitativer RAG-Check: thematisch oder rhetorisch aehnlich?

Ruft `get_similar_anchors()` fuer einen Beispieltext auf und zeigt, welche Anker tatsaechlich als Kalibrierungsreferenz eingebettet wuerden — prueft die im urspruenglichen Orwell-Index-Analysedokument geaeusserte Sorge, dass Embeddings nach *Thema* statt nach *rhetorischem Stil* clustern.

In [ ]:
SAMPLE_TEXT_PATH = "../data/debug_last_run/02_anonymized_text.txt"

import os
if os.path.isfile(SAMPLE_TEXT_PATH):
    sample_text = open(SAMPLE_TEXT_PATH, encoding="utf-8").read()
    similar = get_similar_anchors(sample_text)
    if not similar:
        print(f"Keine Anker zurueckgegeben (unter MIN_ANCHORS={MIN_ANCHORS}?)")
    else:
        for a in similar:
            print(f"Aehnlichkeit {a['similarity']:.2f} | Orwell {a['orwell_index']:.2f} | {a['domain']}")
            print(f"  {a['excerpt'][:200]}...")
            print()
else:
    print(f"{SAMPLE_TEXT_PATH} nicht gefunden.")